In [4]:
import pandas as pd
from lightgbm import LGBMRegressor

df = pd.read_csv("../data/processed/model_data_monthly.csv")
features = [
    "monthly_sst", "zon_winds", "mer_winds", "humidity", "air_temp",
    "readings", "month_sin", "month_cos",
    "clim_sst", "clim_zon_winds", "clim_mer_winds", "clim_humidity",
    "ssta", "zon_winds_anomaly", "mer_winds_anomaly", "humidity_anomaly",
]

train = df.loc[df["split"] == "train"]
validation = df.loc[df["split"] == "validation"]
test = df.loc[df["split"] == "test"]

if df["target_ssta"].isna().any():
    raise ValueError("target_ssta contains missing values")

model = LGBMRegressor(
    random_state=42,
    verbosity=-1,
    n_estimators=150,
    learning_rate=0.2,
    num_leaves=23,
    max_depth=9,
    min_child_samples=20,
)
model.fit(train[features], train["target_ssta"])

validation_ssta = model.predict(validation[features])
test_ssta = model.predict(test[features])


In [6]:
from sklearn.metrics import f1_score
from pathlib import Path
import sys

# The notebook runs from the src/ directory.
sys.path.insert(0, str(Path.cwd().parent))

from notebooks.preprocessing import create_target

validation_classes = [create_target(value) for value in validation_ssta]
macro_f1 = f1_score(
    validation["target_class"],
    validation_classes,
    average="macro",
    zero_division=0,
)
print("Validation macro F1:", macro_f1)

Validation macro F1: 0.15589758307419407


In [7]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display
from sklearn.metrics import f1_score, mean_squared_error

PARAMETER_SWEEPS = {
    "n_estimators": [50, 100, 150, 200, 300, 400],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    "num_leaves": [7, 15, 31, 47, 63],
    "max_depth": [3, 5, 7, 9, 12],
    "min_child_samples": [5, 10, 20, 40, 80],
}

controls = {
    "n_estimators": widgets.IntSlider(value=100, min=50, max=500, step=50, description="Trees", continuous_update=False),
    "learning_rate": widgets.FloatSlider(value=0.1, min=0.01, max=0.3, step=0.01, description="Rate", readout_format=".2f", continuous_update=False),
    "num_leaves": widgets.IntSlider(value=31, min=7, max=63, step=4, description="Leaves", continuous_update=False),
    "max_depth": widgets.IntSlider(value=7, min=3, max=12, step=1, description="Depth", continuous_update=False),
    "min_child_samples": widgets.IntSlider(value=20, min=5, max=100, step=5, description="Min child", continuous_update=False),
}
run_button = widgets.Button(description="Run model", button_style="primary", icon="play")
reset_button = widgets.Button(description="Reset", icon="refresh")
sweep_parameter = widgets.Dropdown(options=list(PARAMETER_SWEEPS), value="n_estimators", description="Sweep:")
sweep_button = widgets.Button(description="Plot sweep", button_style="info", icon="line-chart")
run_output = widgets.Output()
sweep_output = widgets.Output()
history = []


def current_parameters():
    return {name: control.value for name, control in controls.items()}


def evaluate(parameters):
    candidate = LGBMRegressor(random_state=42, verbosity=-1, **parameters)
    candidate.fit(train[features], train["target_ssta"])
    predictions = candidate.predict(validation[features])
    predicted_classes = [create_target(value) for value in predictions]
    macro_f1 = f1_score(
        validation["target_class"],
        predicted_classes,
        average="macro",
        zero_division=0,
    )
    rmse = np.sqrt(mean_squared_error(validation["target_ssta"], predictions))
    return candidate, predictions, macro_f1, rmse


def draw_diagnostics(candidate, predictions, macro_f1, rmse, parameters):
    history.append({**parameters, "macro_f1": macro_f1, "rmse": rmse})
    history_df = pd.DataFrame(history)
    importances = pd.Series(candidate.feature_importances_, index=features).sort_values().tail(10)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes[0, 0].plot(history_df.index + 1, history_df["macro_f1"], marker="o", label="Macro F1")
    axes[0, 0].set_title("Validation Macro F1 by run")
    axes[0, 0].set_xlabel("Run")
    axes[0, 0].set_ylabel("Macro F1")
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].barh(importances.index, importances.values)
    axes[0, 1].set_title("Current feature importance")
    axes[0, 1].set_xlabel("Importance")

    axes[1, 0].scatter(validation["target_ssta"], predictions, s=12, alpha=0.25)
    limits = [validation["target_ssta"].min(), validation["target_ssta"].max()]
    axes[1, 0].plot(limits, limits, "r--", label="Perfect prediction")
    axes[1, 0].set_title("Validation: actual vs predicted SSTA")
    axes[1, 0].set_xlabel("Actual SSTA")
    axes[1, 0].set_ylabel("Predicted SSTA")
    axes[1, 0].legend()

    axes[1, 1].hist(validation["target_ssta"] - predictions, bins=30, edgecolor="white")
    axes[1, 1].set_title("Validation residuals")
    axes[1, 1].set_xlabel("Actual - predicted SSTA")
    axes[1, 1].set_ylabel("Count")

    fig.suptitle(f"Macro F1: {macro_f1:.4f} | RMSE: {rmse:.4f}", fontsize=14)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    display(history_df.sort_values("macro_f1", ascending=False).head(10))


def run_model(_=None):
    parameters = current_parameters()
    candidate, predictions, macro_f1, rmse = evaluate(parameters)
    with run_output:
        clear_output(wait=True)
        draw_diagnostics(candidate, predictions, macro_f1, rmse, parameters)


def reset_model(_=None):
    defaults = {"n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31, "max_depth": 7, "min_child_samples": 20}
    for name, value in defaults.items():
        controls[name].value = value
    history.clear()
    with run_output:
        clear_output(wait=True)
    with sweep_output:
        clear_output(wait=True)


def plot_parameter_sweep(_=None):
    parameter = sweep_parameter.value
    base_parameters = current_parameters()
    rows = []
    for value in PARAMETER_SWEEPS[parameter]:
        parameters = {**base_parameters, parameter: value}
        _, _, macro_f1, rmse = evaluate(parameters)
        rows.append({parameter: value, "macro_f1": macro_f1, "rmse": rmse})
    sweep_df = pd.DataFrame(rows)

    with sweep_output:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].plot(sweep_df[parameter], sweep_df["macro_f1"], marker="o")
        axes[0].set_title(f"Macro F1 vs {parameter}")
        axes[0].set_xlabel(parameter)
        axes[0].set_ylabel("Validation Macro F1")
        axes[0].grid(alpha=0.3)
        axes[1].plot(sweep_df[parameter], sweep_df["rmse"], marker="o", color="darkorange")
        axes[1].set_title(f"RMSE vs {parameter}")
        axes[1].set_xlabel(parameter)
        axes[1].set_ylabel("Validation RMSE")
        axes[1].grid(alpha=0.3)
        fig.tight_layout()
        display(fig)
        plt.close(fig)
        display(sweep_df)

run_button.on_click(run_model)
reset_button.on_click(reset_model)
sweep_button.on_click(plot_parameter_sweep)

parameter_box = widgets.VBox(list(controls.values()))
button_box = widgets.HBox([run_button, reset_button])
sweep_box = widgets.HBox([sweep_parameter, sweep_button])
display(widgets.VBox([parameter_box, button_box, run_output, sweep_box, sweep_output]))
run_model()
